# 05 - Ingestão Silver: ecommerce_clientes
**Squad 2 - Grupo 2**

## Objetivo
Ler o micro-lote novo da Bronze, aplicar limpeza e as regras de qualidade de `ecommerce_clientes`, adicionar `silver_processed_at` e gravar em Delta (append). Linhas que quebram alguma regra vão para uma quarentena, com o motivo.

## Fluxo
`grupo2/bronze/ecommerce_clientes` → limpeza e validações → `grupo2/silver/ecommerce_clientes` (válidos) e `grupo2/quarantine/ecommerce_clientes` (rejeitados), com controle em `grupo2/metadata/silver_control_log`.

## Regras de qualidade aplicadas
| Regra | Descrição | Tratamento (`motivo_rejeicao`) |
|---|---|---|
| 1 | Schema completo (`id_cliente`, `uuid_cliente`, `nome`, `sobrenome`, `email`, `dt_cadastro`) | Coluna ausente bloqueia o lote; campo nulo vai para quarentena (`CAMPO_OBRIGATORIO_NULO`) |
| 2 | E-mail único no lote e inexistente na Silver | Quarentena (`EMAIL_DUPLICADO_NO_LOTE`, `EMAIL_EXISTE_NA_SILVER`) |
| 3 | `dt_cadastro` parseável e não futura | Quarentena (`DT_CADASTRO_INVALIDA`, `DT_CADASTRO_FUTURA`) |
| 4 | `uuid_cliente` em formato UUID v4 | Quarentena (`UUID_INVALIDO`) |
| 5 | Deduplicação de `id_cliente` e e-mail no lote | `id_cliente` repetido: fica o mais recente; e-mail em ids diferentes: quarentena |
| 8 | Alerta: mesmo e-mail em mais de um `id_cliente` (lote ou Silver) | Alerta impresso e quarentena (mesmos motivos da regra 2) |
| 10 | Alerta: `dt_ultima_atualizacao` anterior a `dt_cadastro` | Alerta impresso e quarentena (`DT_ATUALIZACAO_ANTERIOR_AO_CADASTRO`) |

## KPIs gerados
| Regra | KPI | Observação |
|---|---|---|
| 6 | Novos clientes na Silver nos últimos 10 minutos | Medido pelo horário de processamento (`silver_processed_at`) |
| 7 | Cadastros por provedor de e-mail | Alerta se mais de 30% vierem de provedores fora de gmail, yahoo, hotmail e outlook |
| 9 | Tempo médio entre cadastro e primeira compra | **PROVISÓRIO**: pedidos lidos da RAW (somente leitura). Trocar pela Silver de pedidos quando estiver acessível |

## Saídas
- **Silver:** colunas da Bronze (sem as de partição) mais `silver_processed_at`
- **Quarentena:** linhas rejeitadas com `motivo_rejeicao`
- **Log de controle:** arquivos da Bronze já processados (evita reprocessar)

## Como executar
Rodar as células em ordem. Se não houver arquivo novo na Bronze, o notebook encerra na leitura do micro-lote. Nesse caso, para só consultar o resultado, rode as células de imports, caminhos e parâmetros e depois as de visualização e conferência.

In [0]:
%run ./00_setup_config.ipynb

## Importação das bibliotecas
Bibliotecas do Spark e utilitários usados nas etapas seguintes.

In [0]:
# Importação das bibliotecas necessárias

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import os

## Credenciais e caminhos
Monta as opções OAuth do Spark a partir do `.env` e define os caminhos das camadas no ADLS (Bronze, Silver, quarentena e log de controle). Nenhuma credencial é exibida.

In [0]:
# Credenciais OAuth do Spark e caminhos das camadas no ADLS

storage_account_name = "internshipdatalake"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": (
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token"
    )
}

base_grupo = f"abfss://squad2@{storage_account_name}.dfs.core.windows.net/grupo2"

caminho_bronze = f"{base_grupo}/bronze/ecommerce_clientes"
caminho_silver = f"{base_grupo}/silver/ecommerce_clientes"
caminho_quarentena = f"{base_grupo}/quarantine/ecommerce_clientes"
caminho_control_log_silver = f"{base_grupo}/metadata/silver_control_log"

print(f"Bronze     : {caminho_bronze}")
print(f"Silver     : {caminho_silver}")
print(f"Quarentena : {caminho_quarentena}")
print(f"Log Silver : {caminho_control_log_silver}")

## Parâmetros das regras
Concentra os valores usados nas validações (colunas obrigatórias, regex de UUID v4, provedores conhecidos e limite do KPI 7), para ajustar as regras em um só lugar. Inclui a função que verifica se uma tabela Delta já existe.

In [0]:
# Parâmetros das regras de qualidade

COLUNAS_OBRIGATORIAS = ["id_cliente", "uuid_cliente", "nome", "sobrenome", "email", "dt_cadastro"]  # regra 1

# UUID v4: 32 hex + hífens, versão 4 e variante 8/9/a/b (regra 4)
REGEX_UUID_V4 = r"^[0-9a-f]{8}-[0-9a-f]{4}-4[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$"

PROVEDORES_CONHECIDOS = ["gmail", "yahoo", "hotmail", "outlook"]  # regra 7
LIMITE_PROVEDOR_DESCONHECIDO = 0.30  # alerta se mais de 30% dos cadastros vierem de provedor desconhecido

def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

print("Parâmetros e funções prontos.")

## Leitura do micro-lote
O micro-lote são os arquivos da Bronze que ainda não estão no log de controle da Silver. Se não houver nenhum, o notebook encerra.

In [0]:
# Lê o micro-lote: arquivos da Bronze que ainda não estão no log de controle da Silver

df_bronze = spark.read.format("delta").options(**adls_options).load(caminho_bronze)

if tabela_delta_existe(caminho_control_log_silver):
    df_ja_processados = (spark.read.format("delta").options(**adls_options)
        .load(caminho_control_log_silver)
        .select(F.col("arquivo_processado").alias("bronze_source_file"))
        .distinct())
    df_lote = df_bronze.join(df_ja_processados, "bronze_source_file", "left_anti")
else:
    df_lote = df_bronze

resumo_arquivos = df_lote.groupBy("bronze_source_file").agg(F.count("*").alias("linhas")).collect()

if not resumo_arquivos:
    dbutils.notebook.exit("Nenhum micro-lote novo na Bronze para 'ecommerce_clientes'.")

total_lido = sum(r["linhas"] for r in resumo_arquivos)
print(f"Micro-lote: {len(resumo_arquivos)} arquivo(s), {total_lido} linha(s) lidas da Bronze.")

## Regra 1: schema completo
Se faltar alguma coluna obrigatória na Bronze, o lote inteiro é bloqueado com erro, antes de qualquer gravação.

In [0]:
# Regra 1: schema completo. Coluna obrigatória ausente bloqueia o lote inteiro

colunas_faltantes = [c for c in COLUNAS_OBRIGATORIAS if c not in df_lote.columns]

if colunas_faltantes:
    raise ValueError(f"[LOTE BLOQUEADO] Colunas obrigatórias ausentes na Bronze: {colunas_faltantes}")

print("Schema completo: OK")

## Limpeza e validações linha a linha (regras 1, 3, 4 e 10)
Padroniza o texto (espaços, vazio como nulo, e-mail e UUID em minúsculas), converte tipos e atribui a cada linha inválida o primeiro motivo de rejeição encontrado. Linhas sem motivo seguem para as próximas etapas.

In [0]:
# Limpeza e validações linha a linha (regras 1, 3, 4 e 10)

colunas_bronze = df_lote.columns

df_t = df_lote

# Texto: remove espaços, transforma vazio em nulo; e-mail e uuid em minúsculas
for c in ["uuid_cliente", "nome", "sobrenome", "email"]:
    texto = F.trim(F.col(c).cast("string"))
    df_t = df_t.withColumn(c, F.when(F.length(texto) == 0, None).otherwise(texto))

df_t = (df_t
    .withColumn("email", F.lower("email"))
    .withColumn("uuid_cliente", F.lower("uuid_cliente"))
    .withColumn("id_cliente", F.expr("try_cast(id_cliente as bigint)"))
    .withColumn("dt_cadastro_ts", F.expr("try_to_timestamp(dt_cadastro)"))
    .withColumn("dt_atualizacao_ts", F.expr("try_to_timestamp(dt_ultima_atualizacao)"))
)

campo_obrigatorio_nulo = (
    F.col("id_cliente").isNull() | F.col("uuid_cliente").isNull() | F.col("nome").isNull()
    | F.col("sobrenome").isNull() | F.col("email").isNull() | F.col("dt_cadastro").isNull()
)

df_t = df_t.withColumn(
    "motivo_rejeicao",
    F.when(campo_obrigatorio_nulo, F.lit("CAMPO_OBRIGATORIO_NULO"))                                   # regra 1
     .when(F.col("dt_cadastro_ts").isNull(), F.lit("DT_CADASTRO_INVALIDA"))                           # regra 3
     .when(F.col("dt_cadastro_ts") > F.current_timestamp(), F.lit("DT_CADASTRO_FUTURA"))              # regra 3
     .when(~F.col("uuid_cliente").rlike(REGEX_UUID_V4), F.lit("UUID_INVALIDO"))                       # regra 4
     .when(F.col("dt_atualizacao_ts") < F.col("dt_cadastro_ts"), F.lit("DT_ATUALIZACAO_ANTERIOR_AO_CADASTRO"))  # regra 10
)

df_rej_linha = df_t.filter(F.col("motivo_rejeicao").isNotNull())
df_ok = df_t.filter(F.col("motivo_rejeicao").isNull())

print("Validações linha a linha aplicadas.")

## Deduplicação no lote (regras 2, 5 e 8)
Primeiro remove repetições de `id_cliente`, mantendo o registro mais recente. Depois separa para a quarentena os e-mails usados por mais de um `id_cliente` no mesmo lote, já que não há como saber qual cliente é o correto.

In [0]:
# Regra 5: deduplicação de id_cliente dentro do lote (mantém o registro mais recente)

w_id = Window.partitionBy("id_cliente").orderBy(
    F.col("dt_atualizacao_ts").desc_nulls_last(),
    F.col("bronze_ingested_at").desc()
)

df_ok = df_ok.withColumn("_rn", F.row_number().over(w_id))
df_duplicados_id = df_ok.filter(F.col("_rn") > 1)
df_ok = df_ok.filter(F.col("_rn") == 1).drop("_rn")

# Regras 2, 5 e 8: mesmo e-mail em mais de um id_cliente no lote -> quarentena de todos os envolvidos

w_email = Window.partitionBy("email")

df_ok = df_ok.withColumn("_qtd_ids_email", F.size(F.collect_set("id_cliente").over(w_email)))

df_rej_email_lote = (df_ok.filter(F.col("_qtd_ids_email") > 1)
    .withColumn("motivo_rejeicao", F.lit("EMAIL_DUPLICADO_NO_LOTE")))

df_ok = df_ok.filter(F.col("_qtd_ids_email") <= 1).drop("_qtd_ids_email")

print("Deduplicação no lote aplicada.")

## Comparação com a Silver existente (regras 2 e 8)
Descarta linhas cujo `id_cliente` já está na Silver (reenvio de arquivo) e envia para a quarentena as que usam um e-mail já pertencente a outro cliente. Na primeira carga a Silver ainda não existe e esta etapa é pulada.

In [0]:
# Regras 2 e 8: comparação com a Silver já existente

if tabela_delta_existe(caminho_silver):
    df_silver_ref = (spark.read.format("delta").options(**adls_options)
        .load(caminho_silver)
        .select("id_cliente", "email")
        .distinct())

    # Mesmo id_cliente já está na Silver: reenvio de arquivo, descarta sem quarentena
    df_reenvio = df_ok.join(df_silver_ref.select("id_cliente"), "id_cliente", "left_semi")
    df_ok = df_ok.join(df_silver_ref.select("id_cliente"), "id_cliente", "left_anti")

    # E-mail já usado por OUTRO id_cliente na Silver: quarentena
    ids_conflito = (df_ok
        .join(df_silver_ref.withColumnRenamed("id_cliente", "id_cliente_silver"), "email")
        .filter(F.col("id_cliente") != F.col("id_cliente_silver"))
        .select("id_cliente")
        .distinct())

    df_rej_email_silver = (df_ok.join(ids_conflito, "id_cliente", "left_semi")
        .withColumn("motivo_rejeicao", F.lit("EMAIL_EXISTE_NA_SILVER")))
    df_ok = df_ok.join(ids_conflito, "id_cliente", "left_anti")
else:
    df_reenvio = df_ok.limit(0)
    df_rej_email_silver = df_ok.limit(0).withColumn("motivo_rejeicao", F.lit("EMAIL_EXISTE_NA_SILVER"))
    print("Silver ainda não existe: primeira carga.")

print("Comparação com a Silver aplicada.")

## Montagem da Silver e da quarentena
Reúne todas as linhas rejeitadas em um único DataFrame de quarentena e monta a Silver final com os tipos corretos e a coluna `silver_processed_at`.

In [0]:
# Monta a quarentena e a Silver final (silver_processed_at)

def para_quarentena(df):
    return df.select(*colunas_bronze, "motivo_rejeicao")

df_quarentena = (para_quarentena(df_rej_linha)
    .unionByName(para_quarentena(df_rej_email_lote))
    .unionByName(para_quarentena(df_rej_email_silver))
    .withColumn("silver_processed_at", F.current_timestamp()))

df_silver = (df_ok.select(
        "id_cliente", "uuid_cliente", "nome", "sobrenome", "email", "senha_hash",
        F.col("dt_cadastro_ts").alias("dt_cadastro"),
        F.col("dt_atualizacao_ts").alias("dt_ultima_atualizacao"),
        "bronze_source_file", "bronze_ingested_at")
    .withColumn("silver_processed_at", F.current_timestamp()))

print("DataFrames Silver e quarentena montados.")

## Resumo, alertas e KPI 7
Imprime o resumo do lote, os alertas das regras 8 e 10 e a distribuição de cadastros por provedor (KPI 7). Tudo é calculado antes da gravação, porque os DataFrames consultam a Silver atual e o resultado mudaria depois do append.

In [0]:
# Resumo, alertas (regras 8 e 10) e KPI 7. Tudo calculado ANTES de gravar,
# porque os DataFrames dependem da Silver atual e mudariam depois do append.

n_silver = df_silver.count()
n_duplicados_id = df_duplicados_id.count()
n_reenvio = df_reenvio.count()
rejeicoes = {r["motivo_rejeicao"]: r["count"] for r in df_quarentena.groupBy("motivo_rejeicao").count().collect()}
n_quarentena = sum(rejeicoes.values())

print("=== RESUMO DO MICRO-LOTE ===")
print(f"Linhas lidas da Bronze        : {total_lido}")
print(f"Duplicados de id_cliente      : {n_duplicados_id} (descartados)")
print(f"Reenvios já presentes na Silver: {n_reenvio} (descartados)")
print(f"Enviadas para quarentena      : {n_quarentena}")
for motivo, qtd in rejeicoes.items():
    print(f"    - {motivo}: {qtd}")
print(f"Linhas para a Silver          : {n_silver}")

print("\n=== ALERTAS ===")
if rejeicoes.get("EMAIL_DUPLICADO_NO_LOTE", 0) or rejeicoes.get("EMAIL_EXISTE_NA_SILVER", 0):
    print("[ALERTA] Regra 8: mesmo e-mail em mais de um id_cliente (lote ou Silver).")
if rejeicoes.get("DT_ATUALIZACAO_ANTERIOR_AO_CADASTRO", 0):
    print("[ALERTA] Regra 10: dt_ultima_atualizacao anterior a dt_cadastro.")
if not any(m in rejeicoes for m in ["EMAIL_DUPLICADO_NO_LOTE", "EMAIL_EXISTE_NA_SILVER", "DT_ATUALIZACAO_ANTERIOR_AO_CADASTRO"]):
    print("Nenhum alerta neste micro-lote.")

# KPI 7: cadastros por provedor de e-mail no micro-lote
if n_silver > 0:
    provedores = {r["provedor"]: r["count"] for r in (df_silver
        .withColumn("provedor", F.regexp_extract("email", r"@([^.]+)", 1))
        .groupBy("provedor").count().collect())}
    print("\n=== KPI 7: CADASTROS POR PROVEDOR ===")
    for prov, qtd in sorted(provedores.items(), key=lambda x: -x[1]):
        print(f"{prov}: {qtd} ({qtd / n_silver:.0%})")
    desconhecidos = sum(q for p, q in provedores.items() if p not in PROVEDORES_CONHECIDOS)
    if desconhecidos / n_silver > LIMITE_PROVEDOR_DESCONHECIDO:
        print(f"[ALERTA] Regra 7: {desconhecidos / n_silver:.0%} dos cadastros vêm de provedores desconhecidos.")

## Gravação
Grava a quarentena e a Silver em Delta (append) e, por último, o log de controle. Se alguma gravação falhar, o log não é atualizado e o lote é reprocessado na próxima execução.

In [0]:
# Grava quarentena, Silver (Delta, append) e, por último, o log de controle.
# O log só é gravado se as duas gravações anteriores funcionarem; se algo falhar,
# o lote é reprocessado na próxima execução (reenvios já na Silver são descartados).

if n_quarentena > 0:
    (df_quarentena.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .save(caminho_quarentena))
    print("    [OK] Quarentena gravada.")

if n_silver > 0:
    (df_silver.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .save(caminho_silver))
    print("    [OK] Silver gravada.")

schema_log = StructType([
    StructField("tabela_origem", StringType(), False),
    StructField("arquivo_processado", StringType(), False),
    StructField("total_linhas", LongType(), False),
    StructField("status", StringType(), False),
    StructField("timestamp_processamento", StringType(), False),
])

agora = str(datetime.now())
log_data = [("ecommerce_clientes", r["bronze_source_file"], int(r["linhas"]), "SUCCESS", agora) for r in resumo_arquivos]

(spark.createDataFrame(log_data, schema=schema_log).write
    .format("delta")
    .options(**adls_options)
    .mode("append")
    .save(caminho_control_log_silver))

print("    [OK] Log de controle da Silver atualizado.")

## KPI 6: novos clientes nos últimos 10 minutos
Conta os clientes gravados na Silver nos últimos 10 minutos, pelo horário de processamento.

In [0]:
# KPI 6: novos clientes na Silver nos últimos 10 minutos (pelo horário de processamento)

df_silver_atual = spark.read.format("delta").options(**adls_options).load(caminho_silver)

novos_10min = df_silver_atual.filter(
    F.col("silver_processed_at") >= F.expr("current_timestamp() - INTERVAL 10 MINUTES")
).count()

print(f"KPI 6: {novos_10min} novo(s) cliente(s) na Silver nos últimos 10 minutos.")
print(f"Total de clientes na Silver: {df_silver_atual.count()}")

## KPI 9 (regra 9): tempo até a primeira compra
**Origem provisória:** lê os `ecommerce_pedidos.parquet` da RAW, somente leitura. Pedidos cancelados não contam como compra, e clientes com compra anterior ao cadastro ficam fora da média e geram alerta.

**Pendência obrigatória:** trocar a origem pela Silver de pedidos quando estiver acessível.

In [0]:
# Regra 9: tempo médio entre dt_cadastro e a primeira compra
# PROVISÓRIO (liberado pelo tech lead): pedidos lidos da camada RAW, somente leitura.
# TODO (obrigatório): trocar a origem pela SILVER de pedidos quando estiver acessível.

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from io import BytesIO
import pandas as pd

df_silver_atual = spark.read.format("delta").options(**adls_options).load(caminho_silver)

credential = ClientSecretCredential(
    tenant_id=os.getenv("ADLS_TENANT_ID"),
    client_id=os.getenv("ADLS_CLIENT_ID"),
    client_secret=os.getenv("ADLS_CLIENT_SECRET")
)

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential
)
fs_raw = service_client.get_file_system_client("raw")

# Localiza os parquets de pedidos na RAW (somente leitura)
arquivos_pedidos = [
    p.name for p in fs_raw.get_paths(path="real-time-data", recursive=True)
    if not p.is_directory and p.name.endswith("ecommerce_pedidos.parquet")
]
print(f"Arquivos de pedidos na RAW: {len(arquivos_pedidos)}")

if not arquivos_pedidos:
    print("Nenhum arquivo de pedidos encontrado na RAW. KPI 9 não calculado.")
else:
    colunas_pedidos = ["id_cliente", "dt_pedido", "status_pedido"]
    pdfs = []
    for caminho in arquivos_pedidos:
        conteudo = fs_raw.get_file_client(caminho).download_file().readall()
        pdfs.append(pd.read_parquet(BytesIO(conteudo), columns=colunas_pedidos))

    df_pedidos = spark.createDataFrame(pd.concat(pdfs, ignore_index=True))

    # Primeira compra de cada cliente (pedido cancelado não conta como compra)
    df_primeira_compra = (df_pedidos
        .filter(F.coalesce(F.lower(F.col("status_pedido")), F.lit("")) != "cancelado")
        .groupBy("id_cliente")
        .agg(F.min(F.expr("try_to_timestamp(dt_pedido)")).alias("dt_primeira_compra")))

    # Junta com os clientes da Silver (só entram clientes que já compraram)
    df_funil = (df_silver_atual.select("id_cliente", "dt_cadastro")
        .join(df_primeira_compra, "id_cliente")
        .withColumn(
            "dias_ate_primeira_compra",
            (F.col("dt_primeira_compra").cast("long") - F.col("dt_cadastro").cast("long")) / 86400
        ))

    total_com_compra = df_funil.count()
    compra_antes_cadastro = df_funil.filter(F.col("dias_ate_primeira_compra") < 0).count()
    media_dias = (df_funil.filter(F.col("dias_ate_primeira_compra") >= 0)
        .agg(F.avg("dias_ate_primeira_compra")).first()[0])

    print("=== KPI 9 (PROVISÓRIO, origem RAW): TEMPO ATÉ A PRIMEIRA COMPRA ===")
    print(f"Clientes da Silver com pelo menos um pedido: {total_com_compra}")
    print(f"Tempo médio entre cadastro e primeira compra: {media_dias:.1f} dia(s)" if media_dias is not None else "Sem dados para calcular a média.")
    if compra_antes_cadastro:
        print(f"[ALERTA] {compra_antes_cadastro} cliente(s) com primeira compra anterior ao cadastro (fora da média).")

## Visualização do resultado
Mostra o schema e os registros da Silver e da quarentena, lidos direto do ADLS.

In [0]:
# Visualização do resultado: Silver e quarentena lidas direto do ADLS
# (se o notebook saiu por falta de micro-lote novo, rode as células de imports e caminhos antes desta)

df_silver_view = spark.read.format("delta").options(**adls_options).load(caminho_silver)

print("=== SILVER ===")
df_silver_view.printSchema()
display(df_silver_view.orderBy(F.col("silver_processed_at").desc()))

if tabela_delta_existe(caminho_quarentena):
    df_quarentena_view = spark.read.format("delta").options(**adls_options).load(caminho_quarentena)

    print("=== QUARENTENA POR MOTIVO ===")
    display(df_quarentena_view.groupBy("motivo_rejeicao").count())

    print("=== QUARENTENA (registros) ===")
    display(df_quarentena_view.orderBy(F.col("silver_processed_at").desc()))
else:
    print("Quarentena ainda não existe (nenhuma linha rejeitada).")

## Conferência da Silver
Confirma que não há duplicados: linhas, `id_cliente` únicos e e-mails únicos devem ter o mesmo valor. Também lista os registros da quarentena com as datas e o motivo.

In [0]:
# Conferência da Silver: sem duplicados de id_cliente e e-mail

print("Linhas           :", df_silver_view.count())
print("id_cliente únicos:", df_silver_view.select("id_cliente").distinct().count())
print("e-mails únicos   :", df_silver_view.select("email").distinct().count())

# Datas da quarentena, para confirmar que as futuras são realmente futuras
if tabela_delta_existe(caminho_quarentena):
    display(df_quarentena_view.select("id_cliente", "email", "dt_cadastro", "dt_ultima_atualizacao", "motivo_rejeicao"))